In [2]:
import pandas as pd
import numpy as np
df=pd.read_csv('../data/quick_commerce_orders.csv')
df.head(3)

,Order ID,Customer ID,Platform,Order Date & Time,Delivery Time (Minutes),Product Category,Order Value (INR),Customer Feedback,Service Rating,Delivery Delay,Refund Requested
0,ORD000001,CUST2824,JioMart,19:29.5,30,Fruits & Vegetables,382,"Fast delivery, great service!",5,No,No
1,ORD000002,CUST1409,Blinkit,54:29.5,16,Dairy,279,Quick and reliable!,5,No,No
2,ORD000003,CUST5506,JioMart,21:29.5,25,Beverages,599,Items missing from order.,2,No,Yes


In [3]:
print("--Data Structure--")
df.info()
print("--Missing values per column--")
print(df.isnull().sum())

--Data Structure--
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 11 columns):
 #   Column                   Non-Null Count   Dtype 
---  ------                   --------------   ----- 
 0   Order ID                 100000 non-null  object
 1   Customer ID              100000 non-null  object
 2   Platform                 100000 non-null  object
 3   Order Date & Time        100000 non-null  object
 4   Delivery Time (Minutes)  100000 non-null  int64 
 5   Product Category         100000 non-null  object
 6   Order Value (INR)        100000 non-null  int64 
 7   Customer Feedback        100000 non-null  object
 8   Service Rating           100000 non-null  int64 
 9   Delivery Delay           100000 non-null  object
 10  Refund Requested         100000 non-null  object
dtypes: int64(3), object(8)
memory usage: 8.4+ MB
--Missing values per column--
Order ID                   0
Customer ID                0
Platform                   0
Or

In [5]:
df_clean=df.copy()
numeric_cols_with_nas = df_clean.select_dtypes(include=[np.number]).columns
for col in numeric_cols_with_nas:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())
categorical_cols_with_nas = df_clean.select_dtypes(include=['object']).columns
for col in categorical_cols_with_nas:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print("Missing values after treatment:")
print(df_clean.isnull().sum())

Missing values after treatment:
Order ID                   0
Customer ID                0
Platform                   0
Order Date & Time          0
Delivery Time (Minutes)    0
Product Category           0
Order Value (INR)          0
Customer Feedback          0
Service Rating             0
Delivery Delay             0
Refund Requested           0
dtype: int64


In [7]:
date_col = [col for col in df_clean.columns if 'date' in col.lower() or 'time' in col.lower()][0]
print(f"Targeting date column: {date_col}")
df_clean[date_col] = pd.to_datetime(df_clean[date_col], errors='coerce', format='mixed')
df_clean = df_clean.dropna(subset=[date_col])
df_clean['Order_Hour'] = df_clean[date_col].dt.hour
df_clean['Day_of_Week'] = df_clean[date_col].dt.day_name()
df_clean['Is_Weekend'] = df_clean['Day_of_Week'].isin(['Saturday', 'Sunday']).astype(int)
print("Successfully processed timestamps!")
df_clean[['Order_Hour', 'Day_of_Week', 'Is_Weekend']].head()

Targeting date column: Order Date & Time
Successfully processed timestamps!


,Order_Hour,Day_of_Week,Is_Weekend
0,19,Friday,0
2,21,Friday,0
3,19,Friday,0
6,22,Friday,0
9,8,Friday,0


In [10]:
val_col = 'Order Value (INR)'
del_time_col = 'Delivery Time (Minutes)'
df_clean['SLA_Breach'] = (df_clean[del_time_col] > 15).astype(int)
df_clean['Delayed_and_Refunded'] = ((df_clean['SLA_Breach'] == 1) & (df_clean['Refund Requested'] == 'Yes')).astype(int)
df_clean[['Delivery Time (Minutes)', 'SLA_Breach', 'Refund Requested', 'Delayed_and_Refunded']].head()

,Delivery Time (Minutes),SLA_Breach,Refund Requested,Delayed_and_Refunded
0,30,1,No,0
2,25,1,Yes,1
3,42,1,Yes,1
6,22,1,No,0
9,51,1,Yes,1


In [11]:
import os
os.makedirs('../data/processed', exist_ok=True)
df_clean.to_csv('../data/processed/quick_commerce_cleaned.csv', index=False)
print("Success! Your cleaned file is saved at: data/processed/quick_commerce_cleaned.csv")

Success! Your cleaned file is saved at: data/processed/quick_commerce_cleaned.csv
